# Rhythm QC correction — development revision
Uses the existing CardioScore campaign Drive cache. These windows informed this fix and are development regression evidence; AI review is not human ground truth. Frozen criteria and labels remain unchanged. Run in order in a fresh Colab runtime.


In [ ]:
#@title Reuse cache and preserve the frozen baseline
from pathlib import Path
import subprocess, sys, json, hashlib
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CardioScore_validation_campaign_v1')
REPO = Path('/content/CardioScore_rhythm_qc_fix')
BASE = 'cba05082623ad6e0e96ab3dad4cdec95e65050c8'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/Virelion-Biotech/Virelion-CardioScore.git',str(REPO)],check=True)
subprocess.run(['git','checkout',BASE],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)+'[dev]'],check=True)
SOURCE=REPO/'virelion_cardioscore/preprocessing/beat_detection.py'
BASELINE_SOURCE=subprocess.check_output(['git','show',BASE+':virelion_cardioscore/preprocessing/beat_detection.py'],cwd=REPO,text=True)
assert hashlib.sha256(BASELINE_SOURCE.encode()).hexdigest()=='5c93983d76dda12b5e4b6b449b1489968298dbeca92c37c47a1bef2fcbc09faa'
SOURCE.write_text(BASELINE_SOURCE)
OUT=ROOT/'results/rhythm_qc_fix_development_v8'
OUT.mkdir(exist_ok=True)
(OUT/'beat_detection_frozen_baseline.py').write_text(BASELINE_SOURCE)
print('BASELINE',BASE,hashlib.sha256(BASELINE_SOURCE.encode()).hexdigest())
print('RAW_CACHE', (ROOT/'results/cardio_pymea_2022/heldout_annotation').exists())


In [ ]:
#@title Confirm the old autocorrelation path on unchanged raw windows
import sys
sys.path.insert(0,str(REPO))
import numpy as np, pandas as pd
from scipy import signal
from virelion_cardioscore.preprocessing.filtering import FilterConfig, filter_trace
import importlib
import virelion_cardioscore.preprocessing.beat_detection as bd
importlib.reload(bd)
detect_beats=bd.detect_beats
SHEET=ROOT/'results/cardio_pymea_2022/heldout_annotation'
manifest=json.loads((SHEET/'sheet_manifest.json').read_text())
baseline=[]
def path_diagnostics(x,fs):
    x=x-np.mean(x); ac=signal.fftconvolve(x,x[::-1],mode='full')[len(x)-1:]/np.dot(x,x)
    lo=int(.5*fs); hi=min(len(ac)-1,int(4*fs))
    reg=ac[lo:hi+1]; peaks,props=signal.find_peaks(reg,prominence=.02)
    widths=signal.peak_widths(reg,peaks)[0]
    if not len(peaks): return {'no_peaks':True}
    strong=int(np.argmax(reg[peaks])); final=strong; chain=[]
    while True:
        period=lo+peaks[final]; half=period/2
        near=[i for i,p in enumerate(peaks) if abs(lo+p-half)<=.08*half and reg[p]>=.05]
        if not near or half<lo*(1-.08): break
        nxt=max(near,key=lambda i:reg[peaks[i]])
        chain.append({'from_s':period/fs,'to_s':(lo+peaks[nxt])/fs,
                      'parent_corr':float(reg[peaks[final]]),'half_corr':float(reg[peaks[nxt]]),
                      'parent_width_ms':widths[final]/fs*1000,'half_width_ms':widths[nxt]/fs*1000})
        final=nxt
    return {'strongest_period_s':(lo+peaks[strong])/fs,'selected_period_s':(lo+peaks[final])/fs,
            'strongest_corr':reg[peaks[strong]],'selected_corr':reg[peaks[final]],
            'strongest_width_ms':widths[strong]/fs*1000,'selected_width_ms':widths[final]/fs*1000,
            'halving_steps':len(chain),'chain':chain}
for w in manifest['windows']:
    p=SHEET/'windows'/(w['window_id']+'.npy')
    assert hashlib.sha256(p.read_bytes()).hexdigest()==w['npy_sha256']
    a=np.load(p,allow_pickle=False); x=filter_trace(a[:,1],manifest['fs_hz'],FilterConfig())
    f=detect_beats(x,manifest['fs_hz'])
    row={'window_id':w['window_id'],'n_beats':f.n_beats,'expected':f.expected_beat_count,
         'qc_rate':f.beat_detection_rate,**path_diagnostics(x,manifest['fs_hz'])}
    baseline.append(row)
pd.DataFrame([{k:v for k,v in r.items() if k!='chain'} for r in baseline]).to_csv(OUT/'baseline_autocorrelation_path.csv',index=False)
(OUT/'baseline_autocorrelation_path.json').write_text(json.dumps(baseline,indent=2))
print(pd.DataFrame([{k:v for k,v in r.items() if k!='chain'} for r in baseline]).to_string(index=False))
# Existing missed-beat stress case: independently estimated count must retain weak genuine beats.
fs=1000.;t=np.arange(0,10,1/fs);weak=np.zeros_like(t)
for tm in np.arange(.5,10,2): weak+=100*np.exp(-(t-tm)**2/(2*.003**2))
for tm in np.arange(1.5,10,2): weak+=8*np.exp(-(t-tm)**2/(2*.003**2))
print('GENUINE_ALTERNATING_WEAK_BEATS',json.dumps(path_diagnostics(weak,fs),indent=2))
print('DETECTOR_BASELINE_CONFIRMED',sum(r['qc_rate']<.7 for r in baseline),'of 40 rejected before exclusions')


In [ ]:
#@title Apply the complete fix and verify all regressions
FILES={"virelion_cardioscore/preprocessing/beat_detection.py":"\"\"\"\nBeat detection on filtered MEA field-potential traces.\n\nFinds beat onsets (spike peaks) in a single-electrode trace using prominence-\nbased peak detection, per the `beat_detection:` block in config/default.yaml.\nDownstream feature extraction consumes the beat indices returned here to compute\nFPD, beat rate, and amplitude. Repolarization STV is computed from consecutive\nper-beat FPD measurements in endpoints.py.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Any, Optional\n\nimport numpy as np\nfrom scipy import signal\n\n\nRHYTHM_PERIOD_RANGE_S = (0.5, 4.0)\nRHYTHM_QC_ESTIMATOR_VERSION = \"slope-autocorrelation-v2\"\n\n\ndef robust_noise_scale(trace: np.ndarray) -> float:\n    \"\"\"1.4826 * MAD of the trace: a spike-insensitive estimate of its typical fluctuation (uV).\"\"\"\n    x = np.asarray(trace, dtype=float)\n    return float(1.4826 * np.median(np.abs(x - np.median(x)))) if x.size else 0.0\n\n\n@dataclass\nclass BeatDetectionConfig:\n    \"\"\"Configuration for single-electrode beat detection.\"\"\"\n\n    method: str = \"peak\"\n    min_prominence_uv: float = 20.0\n    min_distance_ms: float = 250.0\n    refractory_ms: float = 200.0\n    noise_prominence_multiplier: float = 10.0\n    companion_max_amplitude_ratio: float = 0.6\n    companion_max_gap_fraction_of_period: float = 0.5\n\n    @classmethod\n    def from_dict(cls, cfg: dict[str, Any]) -> \"BeatDetectionConfig\":\n        return cls(\n            method=str(cfg.get(\"method\", \"peak\")),\n            min_prominence_uv=float(cfg.get(\"min_prominence_uv\", 20.0)),\n            min_distance_ms=float(cfg.get(\"min_distance_ms\", 250.0)),\n            refractory_ms=float(cfg.get(\"refractory_ms\", 200.0)),\n            noise_prominence_multiplier=float(cfg.get(\"noise_prominence_multiplier\", 10.0)),\n            companion_max_amplitude_ratio=float(cfg.get(\"companion_max_amplitude_ratio\", 0.6)),\n            companion_max_gap_fraction_of_period=float(cfg.get(\"companion_max_gap_fraction_of_period\", 0.5)),\n        )\n\n\n@dataclass\nclass BeatDetectionResult:\n    \"\"\"Beats detected in a single-electrode trace.\"\"\"\n\n    beat_indices: np.ndarray\n    beat_times_s: np.ndarray\n    amplitudes_uv: np.ndarray\n    fs_hz: float\n    duration_s: float\n    candidate_beat_indices: np.ndarray = field(default_factory=lambda: np.array([], dtype=int))\n    expected_beat_count: float | None = None\n    effective_prominence_uv: float | None = None\n    has_companion: np.ndarray = field(default_factory=lambda: np.array([], dtype=bool))\n    n_beats: int = field(init=False)\n    inter_beat_intervals_s: np.ndarray = field(init=False)\n    beat_detection_rate: float = field(init=False)\n\n    def __post_init__(self) -> None:\n        self.n_beats = len(self.beat_indices)\n        if len(self.has_companion) != self.n_beats:\n            self.has_companion = np.zeros(self.n_beats, dtype=bool)\n        if self.n_beats >= 2:\n            self.inter_beat_intervals_s = np.diff(self.beat_times_s)\n        else:\n            self.inter_beat_intervals_s = np.array([])\n        self.beat_detection_rate = self._estimate_detection_rate()\n\n    def _estimate_detection_rate(self) -> float:\n        \"\"\"Estimate detector coverage against an independent rhythm estimate.\n\n        The expected count is estimated from periodicity in the slope of the\n        filtered waveform before thresholded peak detection. This avoids defining the\n        denominator from the same peaks used by the detector, which would make\n        systematic under-detection self-validating. The estimate is a QC\n        heuristic, not validated sensitivity against ground-truth annotations.\n        \"\"\"\n        if self.n_beats == 0:\n            return 0.0\n        if self.expected_beat_count is None or self.expected_beat_count <= 0:\n            return 0.0\n        ratio = self.n_beats / self.expected_beat_count\n        # Symmetric count agreement penalizes both under- and over-detection.\n        return float(np.clip(min(ratio, 1.0 / ratio), 0.0, 1.0))\n\n    @property\n    def beat_rate_bpm(self) -> float:\n        if self.n_beats < 2:\n            return 0.0\n        mean_ibi_s = float(np.mean(self.inter_beat_intervals_s))\n        if mean_ibi_s <= 0:\n            return 0.0\n        return 60.0 / mean_ibi_s\n\n    @property\n    def mean_amplitude_uv(self) -> float:\n        if self.n_beats == 0:\n            return 0.0\n        return float(np.mean(np.abs(self.amplitudes_uv)))\n\n    @property\n    def ibi_variability(self) -> float:\n        \"\"\"Normalized short-term variability of consecutive inter-beat intervals.\"\"\"\n        if len(self.inter_beat_intervals_s) < 2:\n            return 0.0\n        diffs = np.abs(np.diff(self.inter_beat_intervals_s))\n        mean_ibi = float(np.mean(self.inter_beat_intervals_s))\n        if mean_ibi <= 0:\n            return 0.0\n        return float(np.mean(diffs) / mean_ibi)\n\n    @property\n    def stv(self) -> float:\n        \"\"\"Deprecated compatibility alias for IBI variability; not repolarization STV.\"\"\"\n        return self.ibi_variability\n\n\ndef _period_from_autocorrelation(\n    autocorr: np.ndarray, fs_hz: float, lo_s: float, hi_s: float\n) -> float | None:\n    \"\"\"Find a rhythm period, including supported integer subperiods.\"\"\"\n    min_lag = max(1, int(round(lo_s * fs_hz)))\n    max_lag = min(len(autocorr) - 1, int(round(hi_s * fs_hz)))\n    if max_lag <= min_lag:\n        return None\n    region = autocorr[min_lag : max_lag + 1]\n    if not np.any(np.isfinite(region)):\n        return None\n    peak_indices, _ = signal.find_peaks(region, prominence=0.02)\n    if len(peak_indices) == 0:\n        best_offset = int(np.nanargmax(region))\n        best_value = float(region[best_offset])\n        if not np.isfinite(best_value) or best_value < 0.10:\n            return None\n        return (min_lag + best_offset) / float(fs_hz)\n    best = int(peak_indices[int(np.nanargmax(region[peak_indices]))])\n    period_samples = min_lag + best\n    while True:\n        shorter = None\n        parent_value = float(autocorr[period_samples])\n        # Jitter can make a three-cycle (or longer) lag the strongest peak.\n        # Require substantial support for these alternatives; weak short-lag\n        # morphology must not become a new rhythm. Retain the existing weaker\n        # half-period allowance so genuine alternating weak beats remain visible.\n        max_divisor = int(period_samples / min_lag * 1.08)\n        for divisor in range(max_divisor, 1, -1):\n            target = period_samples / divisor\n            min_support = max(0.05, 0.5 * parent_value) if divisor > 2 else 0.05\n            near = [\n                int(i)\n                for i in peak_indices\n                if abs((min_lag + int(i)) - target) <= 0.08 * target\n                and region[int(i)] >= min_support\n            ]\n            if near:\n                shorter = min_lag + max(near, key=lambda i: region[i])\n                break\n        if shorter is None:\n            break\n        period_samples = shorter\n    return period_samples / float(fs_hz)\n\n\ndef _estimate_expected_beat_count(\n    trace: np.ndarray,\n    fs_hz: float,\n    duration_s: float,\n) -> float | None:\n    \"\"\"Estimate expected beat count over 15-120 bpm from slope autocorrelation.\n\n    First differencing reduces the contribution of broad repolarization waves\n    and slow baseline structure, which can otherwise supply a misleading\n    half-period correlation peak. This transform is only used for the independent\n    QC rhythm estimate; peak detection still consumes the original filtered trace.\n    Genuine alternating strong/weak spikes retain their shared slope morphology\n    and can still support the shorter period without using detected beat counts.\n    \"\"\"\n    if duration_s <= 0 or len(trace) < max(32, int(fs_hz * 1.0)):\n        return 1.0 if len(trace) > 0 else None\n\n    # No fs multiplier is required: autocorrelation is normalized by energy.\n    # Keep duration_s from the original window when converting period to count.\n    x = np.diff(np.asarray(trace, dtype=float))\n    x = x - np.mean(x)\n    variance = float(np.dot(x, x))\n    if not np.isfinite(variance) or variance <= 0:\n        return None\n\n    autocorr = signal.fftconvolve(x, x[::-1], mode=\"full\")[len(x) - 1 :]\n    autocorr = autocorr / variance\n    period_s = _period_from_autocorrelation(autocorr, fs_hz, *RHYTHM_PERIOD_RANGE_S)\n    return float(duration_s / period_s) if period_s else None\n\n\ndef _detect_polarity_peaks(\n    trace: np.ndarray,\n    prominence_uv: float,\n    min_distance_samples: int,\n    refractory_samples: int,\n) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Return refractory-filtered positive and negative candidate peaks.\"\"\"\n    pos_indices, _ = signal.find_peaks(\n        trace,\n        prominence=prominence_uv,\n        distance=min_distance_samples,\n    )\n    neg_indices, _ = signal.find_peaks(\n        -trace,\n        prominence=prominence_uv,\n        distance=min_distance_samples,\n    )\n\n    def apply_refractory(indices: np.ndarray) -> np.ndarray:\n        kept: list[int] = []\n        last_idx = -refractory_samples - 1\n        for idx in indices:\n            idx = int(idx)\n            if idx - last_idx >= refractory_samples:\n                kept.append(idx)\n                last_idx = idx\n        return np.asarray(kept, dtype=int)\n\n    return apply_refractory(pos_indices), apply_refractory(neg_indices)\n\n\ndef _choose_polarity(\n    trace: np.ndarray,\n    pos_indices: np.ndarray,\n    neg_indices: np.ndarray,\n) -> int:\n    \"\"\"Choose the dominant beat polarity from candidate peak amplitudes.\"\"\"\n    pos_mean_height = float(np.mean(np.abs(trace[pos_indices]))) if len(pos_indices) else 0.0\n    neg_mean_height = float(np.mean(np.abs(trace[neg_indices]))) if len(neg_indices) else 0.0\n    return 1 if pos_mean_height >= neg_mean_height else -1\n\n\n\ndef _suppress_companions(\n    indices: np.ndarray,\n    amplitudes: np.ndarray,\n    fs_hz: float,\n    *,\n    max_amplitude_ratio: float,\n    max_gap_fraction_of_period: float,\n    expected_period_s: float | None,\n) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"Fold a smaller same-polarity peak into the preceding beat when timing and size fit.\"\"\"\n    if (\n        len(indices) == 0\n        or max_amplitude_ratio <= 0\n        or not expected_period_s\n        or expected_period_s <= 0\n    ):\n        return np.asarray(indices, dtype=int), np.asarray(amplitudes, dtype=float), np.zeros(len(indices), dtype=bool)\n    max_gap_samples = max_gap_fraction_of_period * expected_period_s * fs_hz\n    kept_indices = [int(indices[0])]\n    kept_amplitudes = [float(amplitudes[0])]\n    has_companion = [False]\n    for idx, amp in zip(indices[1:], amplitudes[1:], strict=True):\n        gap = int(idx) - kept_indices[-1]\n        ratio = abs(float(amp)) / abs(kept_amplitudes[-1]) if kept_amplitudes[-1] else float(\"inf\")\n        if gap <= max_gap_samples and ratio <= max_amplitude_ratio:\n            has_companion[-1] = True\n            continue\n        kept_indices.append(int(idx))\n        kept_amplitudes.append(float(amp))\n        has_companion.append(False)\n    return np.asarray(kept_indices, dtype=int), np.asarray(kept_amplitudes, dtype=float), np.asarray(has_companion, dtype=bool)\n\ndef detect_beats(\n    trace: np.ndarray,\n    fs_hz: float,\n    config: Optional[BeatDetectionConfig] = None,\n) -> BeatDetectionResult:\n    \"\"\"Detect depolarization peaks in a filtered single-electrode trace.\"\"\"\n    if config is None:\n        config = BeatDetectionConfig()\n\n    trace = np.asarray(trace, dtype=float)\n    if trace.ndim != 1:\n        raise ValueError(f\"detect_beats expects a 1D array, got shape {trace.shape}\")\n    if not np.isfinite(fs_hz) or fs_hz <= 0:\n        raise ValueError(\"fs_hz must be finite and positive\")\n    if not np.all(np.isfinite(trace)):\n        raise ValueError(\"detect_beats requires finite trace values\")\n    if config.method != \"peak\":\n        raise ValueError(f\"Unsupported beat detection method: {config.method!r}\")\n    if config.min_prominence_uv < 0 or config.min_distance_ms <= 0 or config.refractory_ms <= 0:\n        raise ValueError(\"Beat detection thresholds and distances must be positive\")\n    if config.noise_prominence_multiplier < 0:\n        raise ValueError(\"noise_prominence_multiplier must be non-negative\")\n    if config.companion_max_gap_fraction_of_period <= 0:\n        raise ValueError(\"companion_max_gap_fraction_of_period must be positive\")\n    if config.companion_max_amplitude_ratio < 0:\n        raise ValueError(\"companion_max_amplitude_ratio must be non-negative\")\n\n    duration_s = len(trace) / fs_hz\n    min_distance_samples = max(1, int(round(config.min_distance_ms / 1000.0 * fs_hz)))\n    refractory_samples = max(1, int(round(config.refractory_ms / 1000.0 * fs_hz)))\n\n    effective_prominence_uv = max(\n        config.min_prominence_uv, config.noise_prominence_multiplier * robust_noise_scale(trace)\n    )\n    pos_indices, neg_indices = _detect_polarity_peaks(\n        trace,\n        prominence_uv=effective_prominence_uv,\n        min_distance_samples=min_distance_samples,\n        refractory_samples=refractory_samples,\n    )\n    polarity = _choose_polarity(trace, pos_indices, neg_indices)\n    peak_indices = pos_indices if polarity > 0 else neg_indices\n\n    # Preserve a low-prominence candidate set as diagnostic metadata, but do\n    # not use it as a sensitivity denominator because it is not a ground-truth\n    # reference and is vulnerable to noise-induced overcounting.\n    candidate_prominence = config.min_prominence_uv * 0.25\n    candidate_pos, candidate_neg = _detect_polarity_peaks(\n        trace,\n        prominence_uv=candidate_prominence,\n        min_distance_samples=min_distance_samples,\n        refractory_samples=refractory_samples,\n    )\n    candidate_indices = candidate_pos if polarity > 0 else candidate_neg\n\n    raw_indices = np.asarray(peak_indices, dtype=int)\n    raw_amplitudes = trace[raw_indices] if len(raw_indices) else np.array([], dtype=float)\n    expected_beat_count = _estimate_expected_beat_count(trace, fs_hz, duration_s)\n    expected_period_s = (\n        duration_s / expected_beat_count\n        if expected_beat_count and expected_beat_count > 0\n        else None\n    )\n    beat_indices, amplitudes_uv, has_companion = _suppress_companions(\n        raw_indices, raw_amplitudes, fs_hz,\n        max_amplitude_ratio=config.companion_max_amplitude_ratio,\n        max_gap_fraction_of_period=config.companion_max_gap_fraction_of_period,\n        expected_period_s=expected_period_s,\n    )\n    candidate_indices = np.asarray(candidate_indices, dtype=int)\n    beat_times_s = beat_indices / fs_hz\n\n    return BeatDetectionResult(\n        beat_indices=beat_indices,\n        beat_times_s=beat_times_s,\n        amplitudes_uv=amplitudes_uv,\n        fs_hz=float(fs_hz),\n        duration_s=float(duration_s),\n        candidate_beat_indices=candidate_indices,\n        expected_beat_count=expected_beat_count,\n        effective_prominence_uv=float(effective_prominence_uv),\n        has_companion=has_companion,\n    )\n","tests/test_rhythm_qc.py":"\"\"\"Rhythm QC must reject missed beats without mistaking slow structure for beats.\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pytest\n\nfrom virelion_cardioscore.preprocessing.beat_detection import (\n    BeatDetectionConfig,\n    _estimate_expected_beat_count,\n    detect_beats,\n)\n\n\ndef pulse_train(fs, period, duration, *, alternating=False):\n    t = np.arange(0, duration, 1 / fs)\n    x = np.zeros_like(t)\n    for i, tm in enumerate(np.arange(period / 2, duration, period)):\n        amplitude = 8 if alternating and i % 2 else 100\n        x += amplitude * np.exp(-((t - tm) ** 2) / (2 * .006**2))\n    return t, x\n\n\n@pytest.mark.parametrize(\"fs\", [250., 1000.])\n@pytest.mark.parametrize(\"period\", [1.2, 2., 3.2])\ndef test_slow_half_period_structure_does_not_double_expected_count(fs, period):\n    duration = 10 * period\n    t, x = pulse_train(fs, period, duration)\n    # A smooth, non-beat component repeats twice per depolarization cycle.\n    # Its voltage autocorrelation is strong, but its slope is small compared\n    # with the narrow spikes. No peak-detector output is provided to QC.\n    x += 20 * np.cos(4 * np.pi * t / period)\n    expected = _estimate_expected_beat_count(x, fs, duration)\n    assert expected == pytest.approx(duration / period, rel=.03)\n\n\n@pytest.mark.parametrize(\"fs\", [250., 1000.])\n@pytest.mark.parametrize(\"period\", [.6, 1., 1.6])\ndef test_true_weak_alternating_beats_remain_in_independent_expected_count(fs, period):\n    duration = 20 * period\n    _, x = pulse_train(fs, period, duration, alternating=True)\n    expected = _estimate_expected_beat_count(x, fs, duration)\n    assert expected == pytest.approx(duration / period, rel=.03)\n    result = detect_beats(x, fs, BeatDetectionConfig(min_prominence_uv=20))\n    assert result.n_beats == 10\n    assert result.beat_detection_rate < .75\n\n\ndef test_qc_does_not_become_self_validating_when_detector_threshold_is_changed():\n    fs, period, duration = 1000., 1., 20.\n    _, x = pulse_train(fs, period, duration, alternating=True)\n    strict = detect_beats(x, fs, BeatDetectionConfig(min_prominence_uv=20))\n    permissive = detect_beats(x, fs, BeatDetectionConfig(min_prominence_uv=5))\n    assert strict.expected_beat_count == permissive.expected_beat_count\n    assert strict.n_beats == 10 and permissive.n_beats == 20\n    assert strict.beat_detection_rate < .75\n    assert permissive.beat_detection_rate > .95\n\n\n@pytest.mark.parametrize(\"seed,amplitude\", [(11, 800.), (12, 600.)])\ndef test_jitter_does_not_turn_three_cycle_correlation_into_the_beat_period(seed, amplitude):\n    from virelion_cardioscore.preprocessing.filtering import FilterConfig, filter_trace\n    from virelion_cardioscore.validation.signal_truth import make_truth_recording\n\n    truth = make_truth_recording(\n        seed, duration_s=90, amplitude_uv=amplitude, noise_sd_uv=3, bpm=50,\n    )\n    for start in [*range(0, 81, 10), 63]:\n        raw = np.round(truth.trace_uv[start*1000:(start+10)*1000], 2)\n        x = filter_trace(raw, truth.fs_hz, FilterConfig())\n        expected = _estimate_expected_beat_count(x, truth.fs_hz, 10.)\n        assert expected == pytest.approx(10/1.2, rel=.04), start\n\n\n@pytest.mark.parametrize(\"n\", [0, 1, 10, 1000])\ndef test_flat_and_short_inputs_remain_safe(n):\n    x = np.zeros(n)\n    expected = _estimate_expected_beat_count(x, 1000., n/1000.)\n    assert expected is None or np.isfinite(expected)\n    if n >= 1000:\n        assert expected is None\n\n\n@pytest.mark.parametrize(\"polarity\", [-1., 1.])\n@pytest.mark.parametrize(\"offset,scale\", [(0., 1.), (500., 7.)])\ndef test_rhythm_estimate_is_independent_of_voltage_polarity_offset_and_scale(polarity, offset, scale):\n    fs, period, duration = 1000., 2., 20.\n    t, x = pulse_train(fs, period, duration)\n    x += 20*np.cos(4*np.pi*t/period)\n    reference = _estimate_expected_beat_count(x, fs, duration)\n    transformed = _estimate_expected_beat_count(offset+polarity*scale*x, fs, duration)\n    assert transformed == pytest.approx(reference, rel=1e-10)\n"}
for name,content in FILES.items():
    p=REPO/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
import importlib
importlib.reload(bd)
import importlib.util, sys
spec=importlib.util.spec_from_file_location('qc_legacy',OUT/'beat_detection_frozen_baseline.py')
legacy=importlib.util.module_from_spec(spec);sys.modules[spec.name]=legacy;spec.loader.exec_module(legacy)
AI_CSV=ROOT/'results/ai_exploratory_annotations_20261019/AI_ONLY_exploratory_annotations.csv'
ai_hash=hashlib.sha256(AI_CSV.read_bytes()).hexdigest()
assert ai_hash=='dc5371641c9cdef3abaf1980c00713584ffc6ba7cf31f623d5122e5e4691dc43'
import csv
ai={r['window_id']:r for r in csv.DictReader(AI_CSV.open())}
from virelion_cardioscore.preprocessing.endpoints import extract_electrode_features
from virelion_cardioscore.preprocessing.filtering import estimate_noise_sd
rows=[]
for w in manifest['windows']:
    a=np.load(SHEET/'windows'/(w['window_id']+'.npy'))
    x=filter_trace(a[:,1],manifest['fs_hz'],FilterConfig())
    old=legacy.detect_beats(x,manifest['fs_hz']);new=bd.detect_beats(x,manifest['fs_hz'])
    ref=[float(t) for t in ai[w['window_id']]['beat_times_s'].split(';') if t]
    noise=estimate_noise_sd(a[:,1],manifest['fs_hz'])
    rows.append({'window_id':w['window_id'],'ai_uncertain':ai[w['window_id']]['unsure']=='1','n_ai_beats':len(ref),
                 'old_beats':old.n_beats,'new_beats':new.n_beats,'same_indices':np.array_equal(old.beat_indices,new.beat_indices),
                 'old_period_s':10/old.expected_beat_count if old.expected_beat_count else None,
                 'new_period_s':10/new.expected_beat_count if new.expected_beat_count else None,
                 'old_qc_rate':old.beat_detection_rate,'new_qc_rate':new.beat_detection_rate,'noise_sd_uv':noise,
                 'old_reject':noise>25 or old.n_beats<2 or old.beat_detection_rate<.7,
                 'new_reject':noise>25 or new.n_beats<2 or new.beat_detection_rate<.7})
comparison=pd.DataFrame(rows);comparison.to_csv(OUT/'old_vs_revised_all40.csv',index=False)
good=comparison[~comparison.ai_uncertain]
summary={'evidence_role':'development_regression_not_independent_validation','baseline_commit':BASE,
'estimator_version':bd.RHYTHM_QC_ESTIMATOR_VERSION,'ai_annotation_sha256':ai_hash,
'n_all_windows':40,'n_ai_confident_windows':len(good),'n_ai_uncertain_excluded':40-len(good),
'old_reject_confident':int(good.old_reject.sum()),'new_reject_confident':int(good.new_reject.sum()),
'old_reject_exact_count':int((good.old_reject & (good.old_beats==good.n_ai_beats)).sum()),
'new_reject_exact_count':int((good.new_reject & (good.new_beats==good.n_ai_beats)).sum()),
'beat_indices_changed_windows':comparison.loc[~comparison.same_indices,'window_id'].tolist(),
'noisy_unresolved_windows_still_rejected':bool(comparison[comparison.window_id.isin(['w24','w28','w30'])].new_reject.all()),
'remaining_rejected_confident_windows':good.loc[good.new_reject,'window_id'].tolist(),
'qc_threshold_unchanged':.7,'human_validation_complete':False}
(OUT/'fix_summary.json').write_text(json.dumps(summary,indent=2))
print(comparison.to_string(index=False))
print('FIX_SUMMARY',json.dumps(summary,indent=2))
assert hashlib.sha256(AI_CSV.read_bytes()).hexdigest()==ai_hash
proc=subprocess.run([sys.executable,'-m','pytest','-q'],cwd=REPO,text=True,capture_output=True)
(OUT/'full_suite.log').write_text(proc.stdout+'\n'+proc.stderr)
print(proc.stdout[-18000:]);print(proc.stderr[-4000:])
print('FULL_SUITE_RETURN_CODE',proc.returncode)
assert proc.returncode==0,'Full suite failed: inspect full_suite.log'
